# Relatório do Projeto: Reconhecimento de Sinais de Trânsito GTSRB\n\nEste relatório descreve o trabalho desenvolvido no notebook `Report.ipynb`, explicando o objetivo do projeto, os dados utilizados, as técnicas aplicadas, os modelos testados e os principais resultados obtidos.\n\nO projeto foi desenvolvido no âmbito de Visão por Computador e Processamento de Imagem, com foco na classificação automática de sinais de trânsito usando o conjunto de dados GTSRB (*German Traffic Sign Recognition Benchmark*).\n\n\n

## 1. Objetivo\n\nO objetivo principal foi construir e avaliar modelos de Deep Learning capazes de classificar imagens de sinais de trânsito em 43 classes diferentes.\n\nA meta prática do trabalho foi aproximar o desempenho de referência do GTSRB, explorando arquiteturas de redes neuronais convolucionais, técnicas de pré-processamento, balanceamento de dados, aumento de dados, ensemble e transfer learning.\n\nO melhor resultado reportado no desenvolvimento foi de **99.39% de exatidão no conjunto de teste**, obtido com uma arquitetura `RobustCNN` treinada a partir do zero.\n\n\n

## 2. Conjunto de Dados Utilizado\n\nFoi utilizado o conjunto de dados **GTSRB**, composto por imagens reais de sinais de trânsito capturadas em diferentes condições de iluminação, escala, perspetiva, nitidez e ruído.\n\nA organização usada segue o formato esperado pelo `torchvision.datasets.ImageFolder`, onde cada classe corresponde a uma pasta com o respetivo identificador numérico.\n\n| Pasta | Função |\n|---|---|\n| `../datasets/original_train_images` | Imagens originais de treino |\n| `../datasets/train_images` | Subconjunto de treino após split |\n| `../datasets/val_images` | Subconjunto de validação |\n| `../datasets/train_balanced` | Treino balanceado com aumento de dados estático |\n| `../datasets/test_images` | Conjunto de teste final |\n\n\n

## 3. Bibliotecas e Ferramentas\n\nO projeto foi implementado em Python, usando principalmente:\n\n| Biblioteca | Utilização |\n|---|---|\n| `torch` | Definição, treino e avaliação das redes neuronais |\n| `torchvision` | Transforms, datasets, modelos pré-treinados e aumentos de dados |\n| `numpy` | Operações numéricas auxiliares |\n| `matplotlib` | Visualização de imagens, curvas de treino e gráficos |\n| `pandas` | Organização de matrizes e tabelas |\n| `seaborn` | Visualização de matrizes de confusão |\n| `sklearn.metrics` | Cálculo da matriz de confusão |\n| `OpenCV` / `cv2` | Técnicas clássicas de processamento de imagem, como CLAHE |\n| `PIL` | Leitura e manipulação de imagens |\n| `torchinfo` | Inspeção das arquiteturas |\n\nForam também usados ficheiros utilitários locais: `util/vcpi_util.py` e `util/our_util.py`.\n\n\n

## 4. Preparação dos Dados\n\nA preparação dos dados incluiu quatro passos principais:\n\n1. Carregamento das imagens com `datasets.ImageFolder`.\n2. Redimensionamento das imagens para uma resolução fixa, inicialmente `32x32`.\n3. Conversão para tensor com `transforms.ToTensor()`.\n4. Separação entre treino, validação e teste.\n\nFoi dada atenção especial ao conjunto de validação, porque o GTSRB contém imagens sequenciais. Imagens muito parecidas podem aparecer em frames próximas, por isso o split deve evitar que sequências semelhantes fiquem ao mesmo tempo no treino e na validação.\n\n\n

## 5. Análise Exploratória\n\nAntes do treino, o notebook faz uma análise visual e estatística do conjunto de dados:\n\n- visualização de amostras aleatórias de treino, validação e teste;\n- verificação da distribuição das classes;\n- identificação do desbalanceamento do conjunto de dados.\n\nEsta etapa foi importante porque o GTSRB não tem uma distribuição uniforme. Sem correção, o modelo poderia aprender melhor as classes maioritárias e ter pior desempenho nas classes raras.\n\n\n

## 6. Balanceamento e Aumento de Dados\n\nPara lidar com o desbalanceamento, foi criado um conjunto `train_balanced`, onde classes minoritárias foram aumentadas até um número alvo de imagens.\n\nForam usadas técnicas de aumento de dados estático, isto é, novas imagens foram criadas e gravadas em disco antes do treino.\n\nTransformações usadas ou testadas:\n\n- rotações pequenas;\n- translações;\n- alteração de escala;\n- shear;\n- alteração de brilho, contraste e saturação;\n- perspetiva;\n- blur;\n- ruído suave;\n- sharpness e grayscale como opções experimentais.\n\nTambém foram testadas estratégias de **aumento de dados dinâmico**, aplicadas durante o treino com `torchvision.transforms`.\n\n\n

## 7. Modelo Baseline\n\nO primeiro modelo implementado foi uma CNN simples, usada como referência experimental.\n\nA `BaselineCNN` contém:\n\n- duas camadas convolucionais;\n- ativações `ReLU`;\n- `MaxPool2d` para redução espacial;\n- uma camada totalmente ligada final.\n\nEste modelo serviu para estabelecer um ponto de partida. O desempenho reportado para o baseline foi aproximadamente **87.12% no conjunto de teste** na conclusão consolidada do notebook.\n\n\n

## 8. Infraestrutura de Treino\n\nO treino foi feito com uma função genérica `train_model`, responsável por:\n\n- colocar o modelo no dispositivo correto (`CPU` ou `GPU`);\n- executar forward pass e backward pass;\n- atualizar os pesos com o otimizador;\n- calcular loss e exatidão de treino;\n- avaliar em validação após cada época;\n- aplicar scheduler de taxa de aprendizagem;\n- guardar automaticamente o melhor checkpoint com base na perda de validação;\n- parar cedo com `Early_Stopping` quando a validação deixa de melhorar.\n\nForam usados `CrossEntropyLoss`, `Adam`, `ReduceLROnPlateau` e `Early_Stopping`.\n\n\n

## 9. RobustCNN\n\nA principal melhoria do projeto foi a substituição da baseline por uma arquitetura mais robusta, chamada `RobustCNN`.\n\nA arquitetura inclui:\n\n- três blocos convolucionais;\n- maior número de canais (`32`, `64`, `128`);\n- `BatchNorm2d` após convoluções;\n- ativações `ReLU`;\n- `MaxPool2d` para redução espacial;\n- `Dropout` para regularização;\n- **Global Average Pooling**, reduzindo a dependência de uma camada `Flatten` grande.\n\nEsta alteração foi a que trouxe o maior ganho do projeto. Segundo a conclusão do notebook, a passagem de `BaselineCNN` para `RobustCNN` elevou o resultado de **87.12% para 99.39%** no conjunto de teste.\n\n\n

## 10. Experiências com Aumento de Dados Dinâmico\n\nDepois da `RobustCNN`, foram testadas várias variantes de aumento de dados dinâmico:\n\n| Estratégia | Ideia |\n|---|---|\n| Apenas estático | Treino com conjunto de dados já balanceado em disco |\n| Estático + dinâmico | Conjunto de dados balanceado + transforms aleatórias durante treino |\n| Apenas dinâmico | Conjunto de dados com aumento de dados em tempo de treino sem aumentar ficheiros |\n| Dinâmico massivo | Concatenação de múltiplas versões dinâmicas do conjunto de dados |\n\nApesar de algumas destas estratégias melhorarem a exatidão de validação, elas não superaram de forma consistente a `RobustCNN` estática no conjunto de teste.\n\n\n

## 11. Processamento Clássico de Imagem\n\nTambém foram testadas técnicas clássicas de processamento de imagem para tentar melhorar a robustez visual dos sinais.\n\nForam exploradas:\n\n- **CLAHE** no canal de luminância em espaço LAB;\n- normalização por média e desvio-padrão do conjunto de dados;\n- gamma correction para simular variações de iluminação;\n- combinação de CLAHE com aumento de dados dinâmico.\n\nEstas técnicas são teoricamente adequadas para lidar com sombras e iluminação irregular. No entanto, no notebook, as experiências com CLAHE e normalização não superaram claramente o pipeline mais simples com `RobustCNN`.\n\n\n

## 12. Experiência com Resolução\n\nO projeto também testou a diferença entre imagens `32x32` e `48x48`.\n\nA hipótese era que `48x48` poderia preservar mais detalhe visual dos sinais. Contudo, aumentar a resolução também aumenta o custo computacional e pode exigir mais regularização.\n\nNo desenvolvimento registado, a resolução maior não produziu uma melhoria clara suficiente para substituir o melhor resultado obtido com o setup mais simples.\n\n\n

## 13. Ensembles\n\nForam testados ensembles com vários checkpoints treinados em condições diferentes.\n\nDuas estratégias foram usadas:\n\n- **Soft voting**: soma dos logits antes do `argmax`.\n- **Hard voting**: votação por classe prevista.\n\nO objetivo era verificar se modelos treinados com pipelines diferentes cometiam erros complementares. No entanto, o ensemble não superou claramente a melhor `RobustCNN` individual.\n\n\n

## 14. Transfer Learning\n\nForam também testados modelos pré-treinados no ImageNet:\n\n- `ResNet-18`;\n- `EfficientNet-B0`.\n\nA estratégia incluiu uma fase inicial de treino apenas da cabeça de classificação e depois fine-tuning completo.\n\nApesar de serem arquiteturas fortes, os resultados reportados ficaram abaixo da melhor `RobustCNN` treinada do zero. Isto é plausível porque o GTSRB é um problema específico, com imagens pequenas e classes bem definidas.\n\n\n

## 15. Test-Time Augmentation\n\nFoi adicionada uma melhoria adicional ao notebook: **Test-Time Augmentation (TTA)**.\n\nA TTA não altera o treino nem os pesos do modelo. Durante a avaliação, cada imagem de teste é avaliada em várias versões ligeiramente modificadas, e os logits são combinados antes da decisão final.\n\nAs transformações usadas foram conservadoras:\n\n- brilho ligeiramente maior;\n- brilho ligeiramente menor;\n- contraste leve;\n- deslocamentos de 1 pixel em quatro direções.\n\nNão foram usados flips horizontais porque, em sinais de trânsito, uma inversão pode mudar o significado visual ou criar uma imagem pouco realista.\n\n\n

## 16. Resultados Principais\n\n| Modelo / Estratégia | Exatidão no teste reportada |\n|---|---:|\n| BaselineCNN | 87.12% |\n| RobustCNN estática | **99.39%** |\n| RobustCNN estática + dinâmica | 99.00% |\n| RobustCNN apenas dinâmica | 99.07% |\n| RobustCNN dinâmica massiva | 98.80% |\n| CLAHE + normalização | 97.51% |\n| CLAHE + dinâmico | 98.82% |\n| ResNet-18 Transfer Learning | 98.46% |\n| EfficientNet-B0 Transfer Learning | 98.24% |\n| Ensemble soft voting | ~99.22% - 99.28% |\n| Ensemble hard voting | ~99.22% - 99.25% |\n\nO melhor resultado identificado foi a `RobustCNN` treinada a partir do zero com dados estaticamente balanceados.\n\n\n

## 17. Interpretação dos Resultados\n\nA conclusão principal é que a arquitetura teve mais impacto do que as restantes técnicas.\n\nA `BaselineCNN` tinha capacidade limitada, o que se refletiu num desempenho claramente inferior. A `RobustCNN`, ao adicionar mais profundidade, batch normalization, dropout e global average pooling, conseguiu generalizar muito melhor.\n\nAs técnicas adicionais, como aumento de dados dinâmico, CLAHE, ensembles e transfer learning, foram úteis para exploração experimental, mas não ultrapassaram consistentemente o modelo principal.\n\n\n

## 18. Pontos Fortes do Trabalho\n\nO trabalho apresenta vários aspetos positivos:\n\n- pipeline completo de treino, validação e teste;\n- análise da distribuição das classes;\n- criação de conjunto de dados balanceado;\n- comparação entre baseline e modelo melhorado;\n- uso de early stopping e checkpointing;\n- avaliação com matrizes de confusão;\n- comparação com transfer learning;\n- exploração de técnicas clássicas de processamento de imagem;\n- teste de ensembles;\n- adição de TTA para avaliação mais robusta.\n\n\n

## 19. Limitações e Cuidados\n\nExistem alguns pontos que devem ser tratados com cuidado numa entrega final:\n\n1. Algumas células do notebook contêm outputs antigos ou resultados de execuções diferentes. Para uma entrega final, o ideal é fazer uma execução limpa do notebook do início ao fim.\n2. Alguns valores aparecem escritos manualmente em tabelas finais. O mais rigoroso seria calcular todos os resultados diretamente a partir das predições guardadas.\n3. A comparação entre modelos deve garantir que todos usam exatamente o mesmo conjunto de teste e transforms equivalentes.\n4. Para resultados muito altos, pequenas diferenças de seed, split ou checkpoint podem alterar a exatidão final.\n5. A exatidão global deve ser complementada com métricas por classe.\n\n\n

## 20. Melhorias Futuras\n\nComo trabalho futuro, seria importante tornar a avaliação ainda mais rigorosa e reprodutível. Em particular, poderiam ser acrescentadas as seguintes melhorias:\n\n- fixar seeds para reduzir variação entre execuções;\n- calcular precision, recall e F1-score por classe;\n- guardar automaticamente os resultados de cada experiência numa tabela ou ficheiro CSV;\n- analisar visualmente todas as imagens mal classificadas;\n- comparar o modelo com e sem TTA numa execução limpa;\n- testar `AdamW`, `label_smoothing` e schedulers alternativos;\n- construir ensembles ponderados apenas com os modelos mais fortes.\n\n\n

## 21. Conclusão\n\nO projeto desenvolveu um pipeline completo para classificação de sinais de trânsito no conjunto de dados GTSRB. A evolução principal foi a passagem de uma CNN simples para uma arquitetura `RobustCNN`, que aumentou significativamente a capacidade de representação e reduziu overfitting através de batch normalization, dropout e global average pooling.\n\nO melhor resultado reportado foi **99.39% de exatidão no conjunto de teste**, valor bastante elevado para o problema. As experiências adicionais com aumento de dados, CLAHE, ensembles e transfer learning contribuíram para validar escolhas de projeto, mesmo quando não superaram o melhor modelo.\n\nAssim, o trabalho demonstra não só a construção de um modelo de alto desempenho, mas também uma abordagem experimental comparativa, onde diferentes técnicas foram testadas e avaliadas de forma crítica.\n\n\n